# ARIMA Models

Generate ARIMA and automatically selected ARIMA forecasts on the shared chronological test window.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pmdarima import auto_arima
from statsmodels.tsa.arima.model import ARIMA

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"


In [ ]:
stock_data = pd.read_csv(DATA_DIR / "tcs_stock_data_cleaned.csv", parse_dates=["Date"])
stock_data = stock_data.sort_values("Date").reset_index(drop=True)
split_index = int(len(stock_data) * 0.8)
train = stock_data.iloc[:split_index]
test = stock_data.iloc[split_index:].copy()
train_close = train["Close"].to_numpy(dtype=float)
test_close = test["Close"].to_numpy(dtype=float)


In [ ]:
arima_model = ARIMA(train_close, order=(5, 1, 0)).fit()
arima_forecast = np.asarray(arima_model.forecast(steps=len(test)), dtype=float)
arima_results = pd.DataFrame(
    {"Date": test["Date"], "Actual": test_close, "ARIMA_Prediction": arima_forecast}
)
arima_results.to_csv(DATA_DIR / "arima_predictions.csv", index=False)


In [ ]:
selected_model = auto_arima(
    train_close,
    start_p=0,
    start_q=0,
    max_p=5,
    max_q=5,
    d=1,
    seasonal=False,
    error_action="raise",
    suppress_warnings=True,
    stepwise=True,
)
auto_forecast = np.asarray(selected_model.predict(n_periods=len(test)), dtype=float)
auto_results = pd.DataFrame(
    {
        "Date": test["Date"],
        "Actual": test_close,
        "Auto_ARIMA_Prediction": auto_forecast,
    }
)
auto_results.to_csv(DATA_DIR / "auto_arima_predictions.csv", index=False)
selected_model.order


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(test["Date"], test_close, label="Actual")
ax.plot(test["Date"], arima_forecast, label="ARIMA")
ax.plot(test["Date"], auto_forecast, label="Auto ARIMA")
ax.set(title="ARIMA Forecasts vs Actual", xlabel="Date", ylabel="Closing Price (INR)")
ax.legend()
fig.autofmt_xdate()
plt.show()
